# DCA Buy-the-Bear v4 — Progressive Reinvestment

After each 70% sell at ATH, reinvestment resets to 5% of reserve and increases by 6.5% per subsequent red-month buy, capping at 70%.

In [ ]:
import pandas as pd
import requests
import time
import json
import numpy as np
from datetime import datetime, timezone

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: '%.6f' % x)

In [ ]:
# Fetch monthly BTCUSDT klines

def fetch_monthly_klines(symbol='BTCUSDT', limit=500):
    url = 'https://api.binance.com/api/v3/klines'
    all_klines = []
    start_time = 0
    while True:
        params = {'symbol': symbol, 'interval': '1M', 'startTime': start_time, 'limit': limit}
        resp = requests.get(url, params=params)
        data = resp.json()
        if not data or len(data) == 0:
            break
        all_klines.extend(data)
        print(f'Fetched {len(data)} months, up to {datetime.fromtimestamp(data[-1][0]/1000, tz=timezone.utc).strftime("%Y-%m")}')
        if len(data) < limit:
            break
        start_time = data[-1][0] + 1
        time.sleep(0.1)
    return all_klines

klines = fetch_monthly_klines()

In [ ]:
columns = ['timestamp', 'open', 'high', 'low', 'close', 'volume',
           'close_time', 'quote_vol', 'trades', 'taker_buy_base', 'taker_buy_quote', 'ignore']
df = pd.DataFrame(klines, columns=columns)
df['date'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
for c in ['open', 'high', 'low', 'close', 'volume']:
    df[c] = df[c].astype(float)
df = df[['date', 'open', 'high', 'low', 'close', 'volume']].copy()
df = df.sort_values('date').reset_index(drop=True)
print(f'{df["date"].min().strftime("%Y-%m")} -> {df["date"].max().strftime("%Y-%m")} ({len(df)} months)')

In [ ]:
# Progressive reinvestment strategy

btc_held = 0.0
usdt_reserve = 0.0
total_invested = 0.0
highest_close = 0.0
reinvest_pct = 5.0  # starts at 5% after each sell, increments by 6.5% per buy
buy_counter = 0  # buys since last sell

records = []

for i, row in df.iterrows():
    close = row['close']
    month = row['date']
    action = 'nothing'

    # 1. Entry: buy $10 + reinvest X% of reserve on red monthly candle
    if close < row['open']:
        extra = usdt_reserve * (reinvest_pct / 100.0) if usdt_reserve > 0 else 0.0
        total_usdt = 10.0 + extra
        btc_bought = total_usdt / close
        btc_held += btc_bought
        total_invested += 10.0
        usdt_reserve -= extra
        buy_counter += 1
        action = f'buy ${total_usdt:.1f} @ {close:.0f} (${10:.0f}+{reinvest_pct:.0f}%={extra:.1f})'

    # 2. Update highest close
    prev_highest = highest_close
    if close > highest_close:
        highest_close = close

    # 3. Exit: sell 70% at new ATH -> reset reinvestment rate
    if btc_held > 0.000001 and close > prev_highest:
        sell_btc = btc_held * 0.7
        sell_usdt = sell_btc * close
        btc_held -= sell_btc
        usdt_reserve += sell_usdt
        reinvest_pct = 5.0  # reset to base
        buy_counter = 0
        act = f'sell 70% @ {close:.0f} (-{sell_btc:.6f} BTC, +{sell_usdt:.1f} USDT)'
        action = f'{action} | {act}' if 'buy' in action else act
    else:
        # If we didn't sell, advance the reinvestment rate (capped at 70%)
        if reinvest_pct < 70.0:
            reinvest_pct = min(70.0, reinvest_pct + 6.5)

    portfolio_value = btc_held * close + usdt_reserve

    records.append({
        'date': month,
        'close': close,
        'action': action,
        'reinvest_pct': reinvest_pct,
        'btc_held': btc_held,
        'usdt_reserve': usdt_reserve,
        'total_invested': total_invested,
        'portfolio_value': portfolio_value,
    })

results = pd.DataFrame(records)
print(f'Total months: {len(results)}')
print(f'Trades with action: {len(results[results["action"] != "nothing"])}')

In [ ]:
# Summary with metrics

final = results.iloc[-1]
last_close = final['close']
total_pnl = final['portfolio_value'] - final['total_invested']
ret_pct = (final['portfolio_value'] / final['total_invested'] - 1) * 100

# Max drawdown
eq = results['portfolio_value']
running_max = eq.cummax()
dd = (running_max - eq) / running_max
max_dd = dd.max() * 100

# Sharpe
monthly_returns = eq.pct_change().dropna()
monthly_returns = monthly_returns[monthly_returns.index >= 12]
sharpe = (monthly_returns.mean() / monthly_returns.std()) * np.sqrt(12) if monthly_returns.std() > 0 else 0.0

# Profit factor
pos_sum = monthly_returns[monthly_returns > 0].sum()
neg_sum = monthly_returns[monthly_returns < 0].abs().sum()
pf = pos_sum / neg_sum if neg_sum > 0 else float('inf')

# Calmar
annualized_ret = (1 + ret_pct / 100) ** (12 / len(df)) - 1
calmar = annualized_ret / (max_dd / 100) if max_dd > 0 else 0

# Count buys/sells
buys = [r for r in records if 'buy' in r['action']]
sells = [r for r in records if 'sell' in r['action']]

print('='*60)
print('PORTFOLIO SUMMARY — Progressive Reinvestment')
print('='*60)
print(f'Total invested:        {final["total_invested"]:.2f} USDT')
print(f'BTC held:              {final["btc_held"]:.8f} BTC')
print(f'BTC held value:        {final["btc_held"] * last_close:.2f} USDT (at {last_close:.2f})')
print(f'USDT reserve:          {final["usdt_reserve"]:.2f} USDT')
print(f'Portfolio value:       {final["portfolio_value"]:.2f} USDT')
print(f'Total P&L:             {total_pnl:.2f} USDT')
print(f'Return:                {ret_pct:.2f}%')
print(f'Max drawdown:          {max_dd:.2f}%')
print(f'Sharpe ratio:          {sharpe:.2f}')
print(f'Profit factor:         {pf:.2f}')
print(f'Calmar ratio:          {calmar:.2f}')
print(f'Months:                {len(results)}')
print(f'Buy months:            {len(buys)}')
print(f'Sell months:           {len(sells)}')

In [ ]:
# Chart

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})

ax1.fill_between(results['date'], results['total_invested'], results['portfolio_value'],
                 where=results['portfolio_value'] >= results['total_invested'],
                 color='green', alpha=0.15, label='Profit')
ax1.fill_between(results['date'], results['portfolio_value'], results['total_invested'],
                 where=results['portfolio_value'] < results['total_invested'],
                 color='red', alpha=0.15, label='Loss')
ax1.plot(results['date'], results['total_invested'], color='gray', linewidth=1.5, linestyle='--', label='Total Invested')
ax1.plot(results['date'], results['portfolio_value'], color='blue', linewidth=2, label='Portfolio Value')

buys = results[results['action'].str.contains('buy', na=False)]
sells = results[results['action'].str.contains('sell', na=False)]
ax1.scatter(buys['date'], buys['portfolio_value'], color='green', s=30, marker='^', zorder=5, label='Buy')
ax1.scatter(sells['date'], sells['portfolio_value'], color='red', s=50, marker='v', zorder=5, label='Sell 70%')

ax1.set_title('DCA Buy-the-Bear v4 — Progressive Reinvestment (5% -> 6.5%/buy -> cap 70%)', fontsize=13, fontweight='bold')
ax1.set_ylabel('USDT')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax2.fill_between(results['date'], 0, dd, color='red', alpha=0.4)
ax2.plot(results['date'], dd, color='red', linewidth=1)
ax2.axhline(y=-max_dd, color='red', linestyle=':', linewidth=1, alpha=0.7, label=f'Max DD: {max_dd:.1f}%')
ax2.set_ylabel('Drawdown (%)')
ax2.set_xlabel('Date')
ax2.legend(loc='lower left', fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('dca-bt-buy-bot-v4-progressive.png', dpi=150, bbox_inches='tight')
print('Chart saved as dca-bt-buy-bot-v4-progressive.png')
plt.show()

In [ ]:
# All trades

pd.set_option('display.max_rows', None)
with_actions = results[results['action'] != 'nothing'].copy()
with_actions[['date', 'close', 'reinvest_pct', 'action', 'btc_held', 'usdt_reserve', 'total_invested', 'portfolio_value']]